# 02 PCA 기준선과 scGPT 실제 추론

무엇을 하는가: 동일 64세포에서 train-only PCA와 공식 whole-human frozen embedding을 실행하고 환자 holdout 결과를 읽는다. 64세포는 동작 확인용이며 성능 주장은 2000/10000세포 결과로 판단한다.

In [1]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'configs/paths.yaml').exists()
os.chdir(ROOT)
os.environ.update(OMP_NUM_THREADS='8', OPENBLAS_NUM_THREADS='8', MKL_NUM_THREADS='8', TOKENIZERS_PARALLELISM='false')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
import yaml, pandas as pd, numpy as np
PATHS = yaml.safe_load((ROOT / 'configs/paths.yaml').read_text())
DATA = Path(PATHS['data_root'])
def run_script(name, config=None, model='scgpt', args=()):
    cmd = [str(ROOT / f'.venvs/vcell-{model}/bin/python'), '-u', str(ROOT / 'scripts' / name)]
    if config: cmd += ['--config', str(ROOT / 'configs' / config)]
    cmd += list(args)
    print(' '.join(cmd))
    p = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(p.stdout[-16000:])
    if p.returncode:
        print(p.stderr[-16000:])
        p.check_returncode()
    return p
print('code:', ROOT, '\ndata:', DATA)


code: /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver 
data: /mnt/S1/sdata/pubdata/Virtual_Cell/gssong/vcell_liver


In [2]:
print((DATA / 'subsets/smoke.h5ad').stat().st_size)
run_script('02_baseline_annotation.py', 'tutorial_pca_smoke.yaml')
run_script('03_scgpt_annotation.py', 'tutorial_scgpt_smoke.yaml')

3568938
/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/.venvs/vcell-scgpt/bin/python -u /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/scripts/02_baseline_annotation.py --config /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/configs/tutorial_pca_smoke.yaml


{"run_id": "tutorial_pca_smoke", "macro_f1": 0.6944444444444443, "balanced_accuracy": 0.75, "n_test_unknown": 0, "n_evaluated": 12, "selected_C": 0.01, "selected_class_weight": "balanced", "split_counts": {"train": 40, "test": 12, "val": 12}, "label_classes": ["B", "Endothelial", "Fibroblast", "Hepatocyte", "Myeloid", "T/NK"], "shuffled_train_labels_test_macro_f1": 0.2833333333333334}

/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/.venvs/vcell-scgpt/bin/python -u /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/scripts/03_scgpt_annotation.py --config /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/configs/tutorial_scgpt_smoke.yaml


scGPT - INFO - match 23150/25712 genes in vocabulary of size 60697.
{"run_id": "tutorial_scgpt_smoke", "macro_f1": 0.7666666666666666, "balanced_accuracy": 0.8333333333333334, "n_test_unknown": 0, "n_evaluated": 12, "selected_C": 0.01, "selected_class_weight": null, "split_counts": {"train": 40, "test": 12, "val": 12}, "label_classes": ["B", "Endothelial", "Fibroblast", "Hepatocyte", "Myeloid", "T/NK"], "shuffled_train_labels_test_macro_f1": 0.06666666666666667}
{"run_id": "tutorial_scgpt_smoke", "status": "executed", "task": "annotation", "dataset_accession": "GSE149614", "model_name": "scGPT-whole-human frozen CLS + supervised logistic head", "seed": 42, "n_cells": 64, "data_hash": "cbbe84169b7362ac0dc19ab51e6af472d7f7f945203ae45d81de5dd0e33693df", "split_manifest_hash": "b184b34a9603373aa683eb75c5c8402d70ac0499f88d097a21e860345993e45c", "pretraining_overlap": "unknown", "wall_seconds": 6.262408401817083, "peak_cpu_rss_gib": 1.2534751892089844, "peak_process_gpu_gib": 0.5546875, "gpu

CompletedProcess(args=['/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/.venvs/vcell-scgpt/bin/python', '-u', '/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/scripts/03_scgpt_annotation.py', '--config', '/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/configs/tutorial_scgpt_smoke.yaml'], returncode=0, stdout='scGPT - INFO - match 23150/25712 genes in vocabulary of size 60697.\n{"run_id": "tutorial_scgpt_smoke", "macro_f1": 0.7666666666666666, "balanced_accuracy": 0.8333333333333334, "n_test_unknown": 0, "n_evaluated": 12, "selected_C": 0.01, "selected_class_weight": null, "split_counts": {"train": 40, "test": 12, "val": 12}, "label_classes": ["B", "Endothelial", "Fibroblast", "Hepatocyte", "Myeloid", "T/NK"], "shuffled_train_labels_test_macro_f1": 0.06666666666666667}\n{"run_id": "tutorial_scgpt_smoke", "status": "executed", "task": "annotation", "dataset_accession": "GSE149614", "model_name": "scGPT-whole-human frozen CLS + supervised logistic head", "seed": 4

In [3]:
for rid in ['tutorial_pca_smoke', 'tutorial_scgpt_smoke']:
    r = json.loads((ROOT / 'results' / rid / 'run.json').read_text())
    print(rid, 'macro-F1=', r['metrics']['macro_f1'], 'balanced accuracy=', r['metrics']['balanced_accuracy'], 'peak allocated GiB=', r.get('peak_cuda_allocated_gib'), 'wall seconds=', r['wall_seconds'])
    print(pd.read_csv(ROOT / 'results' / rid / 'by_patient.tsv', sep='\t').to_string(index=False))

tutorial_pca_smoke macro-F1= 0.6944444444444443 balanced accuracy= 0.75 peak allocated GiB= None wall seconds= 0.2201196886599064
patient_id  n_cells  macro_f1  balanced_accuracy
     HCC02        6  0.583333           0.666667
     HCC09        6  0.777778           0.833333
tutorial_scgpt_smoke macro-F1= 0.7666666666666666 balanced accuracy= 0.8333333333333334 peak allocated GiB= 0.26019763946533203 wall seconds= 6.262408401817083
patient_id  n_cells  macro_f1  balanced_accuracy
     HCC02        6  0.777778           0.833333
     HCC09        6  0.777778           0.833333


생물학적 해석: 모델이 PCA를 이길 필요는 없다. 지도 분류 head는 train label로 학습하고 C/weight는 validation에서 고른다. test 환자는 모델 선택에 쓰지 않는다. scGPT는 내부 collator가 raw count를 binning하므로 사전에 binning하지 않는다.

실패 해결: OOM이면 config batch_size=1. sequence 변경은 별도 run ID로 기록한다. flash-attention은 필수가 아니며 이 환경은 공식 non-flash 지원 경로다. cache/출력은 데이터 루트에 저장된다.